In [51]:
import pandas as pd
import numpy as np

In [52]:
recipes_df = pd.read_csv('recipes_data.csv')

In [53]:
recipes_df.head()

,title,ingredients,directions,link,source,NER,site
0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"[""bite size shredded rice biscuits"", ""vanilla""...",www.cookbooks.com
1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....",www.cookbooks.com/Recipe-Details.aspx?id=699419,Gathered,"[""cream of mushroom soup"", ""beef"", ""sour cream...",www.cookbooks.com
2,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...",www.cookbooks.com/Recipe-Details.aspx?id=10570,Gathered,"[""frozen corn"", ""pepper"", ""cream cheese"", ""gar...",www.cookbooks.com
3,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...",www.cookbooks.com/Recipe-Details.aspx?id=897570,Gathered,"[""chicken gravy"", ""cream of mushroom soup"", ""c...",www.cookbooks.com
4,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...",www.cookbooks.com/Recipe-Details.aspx?id=659239,Gathered,"[""graham cracker crumbs"", ""powdered sugar"", ""p...",www.cookbooks.com


In [54]:
recipes_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2231142 entries, 0 to 2231141
Data columns (total 7 columns):
 #   Column       Dtype 
---  ------       ----- 
 0   title        object
 1   ingredients  object
 2   directions   object
 3   link         object
 4   source       object
 5   NER          object
 6   site         object
dtypes: object(7)
memory usage: 119.2+ MB


In [55]:
recipes_df.title.nunique()

1312870

In [56]:
recipes_df['source'].value_counts()

source
Gathered     1643098
Recipes1M     588044
Name: count, dtype: int64

In [57]:
recipes_df['directions'].str.len().describe()

count    2.231142e+06
mean     5.051099e+02
std      4.524093e+02
min      5.000000e+00
25%      2.210000e+02
50%      3.710000e+02
75%      6.410000e+02
max      1.497900e+04
Name: directions, dtype: float64

In [58]:
recipes_df['title'] = recipes_df['title'].str.strip().str.lower()

In [59]:
recipes_df = recipes_df.drop_duplicates(subset="title", keep="first")

In [60]:
recipes_df.title.duplicated().sum()

np.int64(0)

In [61]:
recipes_df['title'].str.lower().str.contains("vegan").sum()

5803

In [ ]:
df_sample = recipes_df.groupby('site').apply(lambda x: x.sample(min(100, len(x)), random_state=42)).reset_index(drop=True)

In [28]:
import spacy

# Load spaCy model
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    import os
    os.system("python -m spacy download en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

In [29]:
def extract_ingredient_names(ingredient_text):
    """
    Extract ingredient names using NLP, filtering out adjectives, modifiers, and measurements.
    Example: "2 cups chopped chicken, 1 tbsp minced garlic" → "chicken, garlic"
    """
    if not ingredient_text or not isinstance(ingredient_text, str):
        return ""
    
    # List of common measurement words to exclude
    measurement_words = {
        'cup', 'cups', 'tbsp', 'tsp', 'tablespoon', 'tablespoons', 'teaspoon', 'teaspoons',
        'oz', 'ounce', 'ounces', 'lb', 'lbs', 'pound', 'pounds', 'g', 'gram', 'grams',
        'ml', 'l', 'liter', 'liters', 'pinch', 'pinches', 'dash', 'handful', 'handfuls',
        'slice', 'slices', 'piece', 'pieces', 'clove', 'cloves', 'head', 'heads',
        'stalk', 'stalks', 'can', 'cans', 'jar', 'jars', 'package', 'packages', 'pkg',
        'batch', 'batches', 'box', 'boxes', 'bunch', 'bunches', 'bulb', 'bulbs',
        'sprig', 'sprigs', 'wedge', 'wedges', 'fillet', 'fillets', 'side', 'sides'
    }
    
    # Process with spaCy
    doc = nlp(ingredient_text.lower())
    
    # Extract nouns and proper nouns (ingredients), skip adjectives and measurements
    ingredients = []
    for token in doc:
        # Keep nouns and proper nouns, skip adjectives, numbers, punctuation, stop words, and measurements
        if (token.pos_ in ["NOUN", "PROPN"] and 
            not token.is_stop and 
            token.text not in measurement_words):
            ingredients.append(token.text)
    
    return ", ".join(ingredients) if ingredients else ""

# Test the function
print("Testing extraction:")
print(extract_ingredient_names("2 cups chopped chicken"))
print(extract_ingredient_names("1 tbsp minced garlic, salt and pepper"))
print(extract_ingredient_names("3 oz sliced ham"))
print(extract_ingredient_names("1 pinch of cayenne pepper"))

Testing extraction:
chicken
garlic, salt, pepper
ham
cayenne, pepper


In [30]:
# Create a new column combining ingredients + NER without adjectives
print("Creating clean ingredients column...")

# Combine ingredients and NER columns
df_sample['all_ingredients'] = df_sample['ingredients'].fillna('') + ', ' + df_sample['NER'].fillna('')

# Apply NLP extraction to remove adjectives
df_sample['ingredients_clean'] = df_sample['all_ingredients'].apply(extract_ingredient_names)

print(f"✅ Created 'ingredients_clean' column")
print(f"\nSample results:")
print(df_sample[['ingredients', 'NER', 'ingredients_clean']].head(10))

Creating clean ingredients column...
✅ Created 'ingredients_clean' column

Sample results:
                                         ingredients  \
0  ["4 cups chicken broth", "2 cups long grain wh...   
1  ["1 (12 ounce) package wide egg noodles", "1 t...   
2  ["2 1/4 cups cubed cantaloupe", "1 (.25 ounce)...   
3  ["1/4 cup butter", "1 onion, very finely chopp...   
4  ["1 egg, lightly beaten", "1 quart heavy cream...   
5  ["1 (8 ounce) package fine egg noodles", "4 eg...   
6  ["1/2 cup butter", "1 cup chopped onions", "1/...   
7  ["1 (8 ounce) package linguine pasta", "2 tabl...   
8  ["1/4 cup butter", "2 large red onions, thinly...   
9  ["1 cup uncooked long grain rice", "1/2 cup pi...   

                                                 NER  \
0  ["chicken broth", "butter", "salt", "long grai...   
1  ["onion", "egg noodles", "water", "head cabbag...   
2  ["bread flour", "orange zest", "egg", "salt", ...   
3  ["stalks celery", "peanuts", "onion", "all-pur...   
4  ["cooking

In [31]:
# Save the updated dataframe with the clean ingredients column
df_sample.to_csv('recipes_sample.csv', index=False)
print("✅ Saved updated recipes_sample.csv with 'ingredients_clean' column")

✅ Saved updated recipes_sample.csv with 'ingredients_clean' column


In [42]:
df_sample['ingredients_clean'].str.split(', ').explode().value_counts().head(20).to_dict().keys()

dict_keys(['salt', 'oil', 'pepper', 'sugar', 'butter', 'cheese', 'ground', 'onion', 'flour', 'water', 'juice', 'sauce', 'cream', 'lemon', 'milk', 'garlic', 'chicken', 'eggs', 'powder', 'vinegar'])

In [44]:
df_sample.isnull().sum()

title                0
ingredients          0
directions           0
link                 0
source               0
NER                  0
site                 0
all_ingredients      0
ingredients_clean    0
dtype: int64

In [50]:
df_sample['ingredients_clean'].str.contains('egg').sum()

np.int64(752)

In [46]:
df_sample['ingredients_clean'].str.split(', ').explode().value_counts().head(20)

ingredients_clean
salt       3269
oil        2641
pepper     2455
sugar      2357
butter     1747
cheese     1505
ground     1485
onion      1343
flour      1340
water      1192
juice      1189
sauce      1133
cream      1082
lemon       961
milk        958
garlic      875
chicken     869
eggs        850
powder      846
vinegar     781
Name: count, dtype: int64